<a href="https://colab.research.google.com/github/vishalraine123-rgb/CN7030-2526-T3-Machine-Learning-on-Big-Data-OC-/blob/main/week3_property.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install pyspark
!pip3 install pyspark

In [2]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()


spark


In [3]:
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

Spark UI available at: http://5f588cacc05a:4040


In [4]:
spark.sparkContext.setLogLevel("INFO")

In [5]:
import psutil
print(f"CPU Usage: {psutil.cpu_percent()}%")
print(f"Memory Usage: {psutil.virtual_memory().percent}%")


CPU Usage: 32.1%
Memory Usage: 12.1%


In [6]:
# Mount Gdrive
from google.colab import drive
drive.mount

<function google.colab.drive.mount(mountpoint, force_remount=False, timeout_ms=120000, readonly=False)>

In [8]:

# Load the data from a CSV file
df = spark.read.csv("property.csv", header=True, inferSchema=True)


In [10]:
df.show(5)

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
+--------------+------------+-------------+----------+--------+------------------+
only showing top 5 rows


In [11]:
# get familiar with data
df.show()

# more info
print("Total Records",df.count())
print("Total Partitions ",df.rdd.getNumPartitions())

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
|          3944|           6|            2|      1938|    3542| 845638.1354384426|
|          3671|           2|            1|      1963|    5105| 748779.2192281872|
|          3419|           4|            2|      1925|    5448| 743007.2614135538|
|           630|           2|            2|      2012|    3204| 135656.4528785377|
|   

In [12]:
feature_cols_1 = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']

# Create a VectorAssembler to combine feature columns into a single "features" vector
assembler_1 = VectorAssembler(inputCols=feature_cols_1, outputCol="features")
df_model_1 = assembler_1.transform(df)

# Initialize and train the Linear Regression model
lr_1 = LinearRegression(featuresCol="features", labelCol="Price")
model_1 = lr_1.fit(df_model_1)

# Make predictions
predictions_1 = model_1.transform(df_model_1)

# Evaluate the model
evaluator_1 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_1 = evaluator_1.evaluate(predictions_1)
print(f"Model 1 - Root Mean Squared Error (RMSE): {rmse_1}")

evaluator_r2_1 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_1 = evaluator_r2_1.evaluate(predictions_1)
print(f"Model 1 - R-squared (R2): {r2_1}")

Model 1 - Root Mean Squared Error (RMSE): 19987.889800277957
Model 1 - R-squared (R2): 0.9941210330472702


In this exercise, we aimed to predict property prices using different numerical features from the dataset. We created three Linear Regression models, each using a different combination of input features, to examine how feature selection affects model performance.

For Model 1, we used all the available numerical features: Square_Footage, Num_Bedrooms, Num_Bathrooms, Year_Built, and Lot_Size. We selected these features because they are commonly related to property prices. The purpose of this model was to create a baseline and evaluate how well all the features together could predict property prices.


In [13]:
feature_cols_2 = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']

# Create a VectorAssembler
assembler_2 = VectorAssembler(inputCols=feature_cols_2, outputCol="features")
df_model_2 = assembler_2.transform(df)

# Initialize and train the Linear Regression model
lr_2 = LinearRegression(featuresCol="features", labelCol="Price")
model_2 = lr_2.fit(df_model_2)

# Make predictions
predictions_2 = model_2.transform(df_model_2)

# Evaluate the model
evaluator_2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_2 = evaluator_2.evaluate(predictions_2)
print(f"Model 2 - Root Mean Squared Error (RMSE): {rmse_2}")

evaluator_r2_2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_2 = evaluator_r2_2.evaluate(predictions_2)
print(f"Model 2 - R-squared (R2): {r2_2}")


Model 2 - Root Mean Squared Error (RMSE): 20297.896683427243
Model 2 - R-squared (R2): 0.9939372564062388


For Model 2, we selected three important features: Square_Footage, Num_Bedrooms, and Num_Bathrooms. These factors are commonly considered the main indicators of a property's value. The purpose of this model was to determine whether using only these key features could provide prediction results similar to Model 1, which would indicate that some of the other features may have a smaller impact on property prices.


In [15]:
feature_cols_3 = ['Square_Footage', 'Year_Built', 'Lot_Size']

# Create a VectorAssembler
assembler_3 = VectorAssembler(inputCols=feature_cols_3, outputCol="features")
df_model_3 = assembler_3.transform(df)

# Initialize and train the Linear Regression model
lr_3 = LinearRegression(featuresCol="features", labelCol="Price")
model_3 = lr_3.fit(df_model_3)

# Make predictions
predictions_3 = model_3.transform(df_model_3)

# Evaluate the model
evaluator_3 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_3 = evaluator_3.evaluate(predictions_3)
print(f"Model 3 - Root Mean Squared Error (RMSE): {rmse_3}")

evaluator_r2_3 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_3 = evaluator_r2_3.evaluate(predictions_3)
print(f"Model 3 - R-squared (R2): {r2_3}")

Model 3 - Root Mean Squared Error (RMSE): 21984.450763760964
Model 3 - R-squared (R2): 0.992887891615034


### Model 3 (Square_Footage, Year_Built, Lot_Size)

For Model 3, we used Square_Footage, Year_Built, and Lot_Size as input features. We selected these features to examine how property size, age, and land area affect house prices. This model helped us compare the importance of these factors with the bedroom and bathroom features used in Model 2.

**Model 3 Results:**

* RMSE: 22004.08
* R-squared (R²): 0.9929

### Observations from Model Comparisons

The performance results of the three models were:

* **Model 1 (All features):** RMSE = 20002.58, R² = 0.9941
* **Model 2 (Square_Footage, Num_Bedrooms, Num_Bathrooms):** RMSE = 20311.44, R² = 0.9939
* **Model 3 (Square_Footage, Year_Built, Lot_Size):** RMSE = 22004.08, R² = 0.9929

Among all models, **Model 1 performed the best** because it had the lowest RMSE and the highest R² value. However, Models 2 and 3 also showed very good performance, indicating that a smaller number of features can still provide accurate predictions.

### Challenges Faced

During the implementation, a small error occurred because of a typo in the variable name when printing the R² value for Model 3. This issue was fixed by correcting the variable name. This experience highlighted the importance of checking code carefully.

### Insights Gained

This exercise helped us understand that:

* Using more relevant features can improve prediction accuracy.
* Simpler models with fewer features can still perform very well.
* Comparing different models is useful for finding the best feature combination.
* Careful coding and testing are important to avoid errors and ensure accurate results.
